# Lat1 pull-down — volcano plot (LAT1 bait vs WT control)

Re-analysis of the Lat1 affinity-purification MS data for the PDH–mtDNA manuscript.

## Which sheet, and why

The workbook has three sheets. Only one is the right granularity for a volcano plot:

| Sheet | What it is | Use here |
|---|---|---|
| **#1** | Protein-level, **trimmed** copy. LFQ values are **already log2** (≈20). **Lacks** `Reverse`/`Potential contaminant`/`Only identified by site` columns. | usable, but less transparent |
| **#2** | Full MaxQuant **`proteinGroups.txt`**: one row per protein group, **raw** LFQ intensities, **carries the QC flag columns**. | ✅ **use this** |
| **#3** | Peptide/PSM-level (`Sequence`, `Raw file`, `m/z`, `Retention time`). | ✗ wrong granularity |

We use **Sheet #2** so filtering is explicit and the log2 transform is done here
(reproducible). If you ever want to run on Sheet #1 instead, set
`ALREADY_LOG2 = True` in the config.

## Workflow (Perseus-style, standard for MaxQuant LFQ interactomes)

1. Filter `Reverse` / `Potential contaminant` / `Only identified by site` (+ `CON__`/`REV__` ID prefixes)
2. log2-transform LFQ; `0 → NaN`
3. Keep proteins with **≥ 3** valid values in **at least one** group
4. Impute remaining `NaN` from a **down-shifted normal** (1.8 SD below the column mean, width 0.3 SD; models below-detection absence in the control — appropriate for a bait pull-down)
5. Welch's *t*-test per protein; log2 fold change = mean(LAT1) − mean(WT)
6. Benjamini–Hochberg FDR (*q*)
7. Volcano, coloured by protein class, key proteins labelled

## Design and conventions

- **Samples:** four replicates per condition — `LAT1_1–4` (Lat1-tagged bait) and `WT_1–4` (untagged control). [TODO: biological or technical replicates]
- **Direction:** log2FC = mean(LAT1) − mean(WT), so **positive = enriched with the bait**.
- **Enrichment criteria:** *q* < 0.05 **and** log2FC > 1 (2-fold). The call is one-sided by design — only enrichment is interpretable in a pull-down, so depleted proteins are never flagged.
- **Ranking:** hits are ordered by π-score (log2FC × −log10 *q*), combining effect size and significance.
- **Reproducibility:** imputation is seeded (`SEED = 42`), so q-values and the volcano are identical across runs.

**Caveat on imputation.** Proteins absent from the control receive imputed
values drawn from a narrow low-intensity distribution. Their fold changes
reflect non-detection rather than measured depletion, and the associated
p-values are anticonservative. This is standard for AP-MS interactome data, but
these are often the strongest apparent hits and should be read accordingly.

## Running this notebook

Set `DATA_DIR` in the config cell to the directory containing
`MS Results Nupur Sharma 11.08.2023.xlsx` and `yeast_encyclopedia.pkl`.

**Requires:** `pandas`, `numpy`, `scipy`, `matplotlib`, `openpyxl`;
`adjustText` optional (label de-overlapping, falls back gracefully).


## 1. Setup

Library imports and global figure settings for the Lat1 pull-down analysis.

| Library | Role in this notebook |
|---|---|
| `pandas` | Reading the MaxQuant `proteinGroups` sheet and reshaping intensity data |
| `numpy` | Log transformation, imputation, and array operations |
| `scipy.stats` | Welch's *t*-test and normal distribution sampling for imputation |
| `matplotlib` | Volcano plot generation |
| `matplotlib.lines.Line2D` | Custom legend handles for the protein-class colouring |

**Figure settings** are applied globally so that every panel exports
consistently:

- `svg.fonttype: none` and `pdf.fonttype: 42` keep text as editable text rather
  than outlines, so labels remain selectable and can be adjusted in Illustrator
  or Inkscape during figure assembly.
- `figure.dpi: 110` for on-screen rendering.
- Top and right spines removed; base font size 10 pt.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({"figure.dpi":110, "font.size":10, "svg.fonttype":"none",
                     "pdf.fonttype":42, "axes.spines.top":False, "axes.spines.right":False})

## 2. Configuration

All analysis parameters are defined here so that thresholds and sample
assignments are explicit and adjustable in one place.

**Input.** Sheet #2 of the MaxQuant workbook (full `proteinGroups` table with
raw LFQ intensities and QC flag columns). `ALREADY_LOG2 = False` since the log2
transform is performed in this notebook; set to `True` only when pointing at
the pre-transformed Sheet #1.

**Sample groups.** Four replicates per condition: `LAT1_1–4` (Lat1-tagged bait
pull-down) and `WT_1–4` (untagged control). Log2 fold change is computed as
mean(LAT1) − mean(WT), so **positive values indicate enrichment with the bait**.

**Filtering and significance thresholds:**

| Parameter | Value | Meaning |
|---|---|---|
| `MIN_VALID` | 3 | Minimum valid (non-missing) log2 values required in at least one group |
| `FC_CUT` | 1.0 | Log2 fold change cutoff (2-fold enrichment) |
| `Q_CUT` | 0.05 | Benjamini–Hochberg FDR cutoff |
| `SEED` | 42 | Random seed, fixing the imputation draw for reproducibility |

**Imputation parameters** (Perseus defaults): missing values are drawn from a
normal distribution down-shifted 1.8 SD below the column mean, with a width of
0.3 column SD. This models below-detection absence in the control, which is the
expected pattern for proteins genuinely enriched by a bait pull-down.


In [ ]:
# ============================== CONFIG ==============================
# Path to the MaxQuant results workbook.
DATA_DIR = '.'
FILE   = f"{DATA_DIR}/MS Results Nupur Sharma 11.08.2023.xlsx"
SHEET  = 1                       # Sheet #2 (0-indexed). Full proteinGroups sheet.
ALREADY_LOG2 = False             # True only if you point this at Sheet #1

# sample columns — MUST match the headers exactly.
# LAT1 = bait (Lat1-tagged pull-down); WT = untagged control.
# log2FC is mean(LAT1) - mean(WT), so POSITIVE = enriched with the bait.
# (NB: do not swap these two — swapping inverts the volcano.)
LAT1_COLS = [f"LFQ intensity LAT1_{i}" for i in range(1,5)]   # bait
WT_COLS   = [f"LFQ intensity WT_{i}"   for i in range(1,5)]   # control

GENE_COL  = "Gene names"
ID_COL    = "Protein IDs"

# thresholds
MIN_VALID   = 3      # >= this many valid (non-missing) log2 values in at least one group
FC_CUT      = 1.0    # |log2 fold change| cutoff  (2-fold)
Q_CUT       = 0.05   # BH-FDR cutoff
SEED        = 42     # reproducible imputation

# imputation (Perseus defaults)
IMP_DOWNSHIFT = 1.8  # in SD below the column mean
IMP_WIDTH     = 0.3  # SD of the imputed distribution, in units of column SD
# ===================================================================

## 3. Helper functions

Two utility functions used by the analysis below.

**`eu_to_num`** — parses numeric columns that may be stored as European
decimal-comma strings (e.g. `1234,56`) rather than floats. Non-breaking spaces
and regular spaces are stripped, decimal commas converted to points, and any
value that cannot be parsed is coerced to `NaN`. This guards against silent
type errors when the workbook is exported from software using a European locale.

**`bh_qvalues`** — Benjamini–Hochberg false discovery rate correction,
implemented directly rather than relying on an external dependency. Missing
p-values are excluded from the correction (they do not contribute to the
multiple-testing burden) and returned as `NaN`. Q-values are computed in
reverse rank order with a running minimum, which enforces the monotonicity
required by the step-up procedure, and are capped at 1.

In [ ]:
def eu_to_num(s):
    """Parse either EU decimal-comma strings or plain floats -> float."""
    return pd.to_numeric(
        s.astype(str)
         .str.replace("\u00a0","",regex=False).str.replace(" ","",regex=False)
         .str.replace(",",".",regex=False),
        errors="coerce")

def bh_qvalues(p):
    """Benjamini-Hochberg FDR (manual, NaN-safe)."""
    p = np.asarray(p, float)
    n = int(np.sum(~np.isnan(p)))
    order = np.argsort(np.where(np.isnan(p), np.inf, p))
    q = np.full_like(p, np.nan); prev = 1.0
    for i in range(len(order)-1, -1, -1):
        idx = order[i]; pv = p[idx]
        if np.isnan(pv): continue
        prev = min(prev, pv*n/(i+1)); q[idx] = prev
    return q

In [ ]:
# --- load ---
df = pd.read_excel(FILE, sheet_name=SHEET)
print(f"loaded {df.shape[0]} protein groups, {df.shape[1]} columns")
missing = [c for c in LAT1_COLS+WT_COLS if c not in df.columns]
assert not missing, f"columns not found (check headers/sheet): {missing}"

## 4. Filtering, imputation, and differential enrichment

The core analysis, following the Perseus-style workflow standard for MaxQuant
LFQ interactome data.

**1. QC filtering.** Protein groups flagged by MaxQuant as reverse database
hits, potential contaminants, or identified only by a modification site are
removed, together with any entries whose protein IDs carry the `CON__` or
`REV__` prefixes. The flag check is written defensively so that it returns
`False` for any QC column absent from the sheet, rather than raising.

**2. Log2 transformation.** LFQ intensities are parsed to numeric. Zero and
blank values encode non-detection and are converted to `NaN` before
transformation, so they are treated as missing rather than as measured zeros.

**3. Valid-value filter.** Protein groups are retained if they have at least
`MIN_VALID` (3) non-missing log2 values in **at least one** of the two groups.
Requiring valid values in only one group is deliberate: proteins genuinely
enriched by the bait are expected to be absent from the control, and a
both-groups requirement would discard exactly the class of interest.

**4. Imputation.** Remaining missing values are drawn per column from a normal
distribution down-shifted 1.8 SD below the observed column mean, with width
0.3 column SD. Mean and SD are computed from observed values only. The draw is
seeded (`SEED = 42`), so the imputed values — and therefore all downstream
p-values — are reproducible across runs.

**5–6. Testing.** Welch's *t*-test (unequal variances) is applied per protein
across the four replicates per group. Log2 fold change is the difference of
group means, mean(LAT1) − mean(WT), so positive values indicate enrichment with
the bait. P-values are corrected by Benjamini–Hochberg to give q-values.

**Significance criteria.** A protein is called enriched when q < 0.05 **and**
log2FC > 1 (2-fold). Note this test is one-sided by construction: proteins
depleted relative to control are never flagged, which is appropriate for a bait
pull-down where only enrichment is interpretable.

Gene names are taken as the first entry in each protein group's semicolon-
separated list, upper-cased.

In [ ]:
# --- 1) QC filter: contaminants, reverse hits, identified-by-site-only ---
def flag(col):
    return df[col].astype(str).str.strip().eq("+") if col in df.columns else pd.Series(False, index=df.index)

qc_bad = flag("Reverse") | flag("Potential contaminant") | flag("Only identified by site")
id_bad = df[ID_COL].astype(str).str.startswith(("CON__","REV__"))
d = df[~(qc_bad | id_bad)].copy()
print(f"removed {int((qc_bad|id_bad).sum())} (contaminant/reverse/by-site); {len(d)} remain")

In [ ]:
# --- 2) numeric LFQ + log2 ---
M = d[LAT1_COLS + WT_COLS].apply(eu_to_num)
# a 0 (or blank) encodes "not detected" -> NaN; log2 only if not already log2
L = M.replace(0, np.nan)
if not ALREADY_LOG2:
    L = np.log2(L)

In [ ]:
# --- 3) valid-value filter (>= MIN_VALID in at least one group) ---
vl = L[LAT1_COLS].notna().sum(axis=1)
vw = L[WT_COLS].notna().sum(axis=1)
keep = (vl >= MIN_VALID) | (vw >= MIN_VALID)
d, L = d[keep].reset_index(drop=True), L[keep].reset_index(drop=True)
print(f"{len(d)} proteins pass the valid-value filter")

In [ ]:
# --- 4) impute missing values: down-shifted normal, per column ---
rng = np.random.default_rng(SEED)
Li = L.copy()
for c in Li.columns:
    col = Li[c].astype(float)
    m, s = col.mean(), col.std()
    miss = col.isna().values
    if miss.any():
        Li.loc[miss, c] = rng.normal(m - IMP_DOWNSHIFT*s, IMP_WIDTH*s, int(miss.sum()))

In [ ]:
# --- 5/6) Welch t-test, fold change, BH-FDR ---
a = Li[LAT1_COLS].to_numpy(float)
b = Li[WT_COLS].to_numpy(float)
t, p = stats.ttest_ind(a, b, axis=1, equal_var=False)   # Welch
res = pd.DataFrame({
    ID_COL:  d[ID_COL].values,
    "gene":  d[GENE_COL].astype(str).str.split(";").str[0].str.upper().values,
    "log2FC": a.mean(1) - b.mean(1),      # + = enriched with Lat1 bait
    "p": p,
})
res["q"] = bh_qvalues(res["p"].values)
res["neglog10q"] = -np.log10(res["q"].clip(lower=1e-300))
res["significant"] = (res["q"] < Q_CUT) & (res["log2FC"] > FC_CUT)
print(f"{int(res['significant'].sum())} proteins enriched (q<{Q_CUT}, log2FC>{FC_CUT})")

## 5. Protein classification

Assigns each detected protein to a functional class for colouring the volcano
plot, and flags mitochondrial and mtDNA-encoded products.

**Mitochondrial reference set.** Loaded from the yeast annotation table
(`yeast_encyclopedia.pkl`) using the Morgenstern et al. high-confidence
mitochondrial proteome flag. The column lookup is written defensively: the
annotation table carries MultiIndex columns which may survive pickling as
tuples or as their string representations, so `_norm_col` normalises either
form before matching. The truthiness check likewise accepts booleans, strings,
and numeric encodings, since the flag column's dtype depends on how the table
was built.

**Functional classes.** Proteins are assigned to one of the following, tested
in dictionary order — the **first match wins**, so genes appearing in more than
one list take the class listed earlier:

| Class | Contents |
|---|---|
| PDH complex | Pyruvate dehydrogenase subunits, including the Lat1 bait |
| mtDNA / nucleoid | Nucleoid-associated and mtDNA maintenance proteins |
| Large Ribosomal Subunit | Mitoribosome large subunit |
| Small Ribosomal Subunit | Mitoribosome small subunit |
| TCA | Tricarboxylic acid cycle enzymes |
| Trans. Factors | Mitochondrial translation factors and aminoacyl-tRNA synthetases |
| mtDNA encoded | Products of the mitochondrial genome |
| other | All remaining proteins (grey) |

Mitoribosomal proteins not explicitly listed are caught by a prefix fallback
(`MRPL` → large; `MRPS`, `RSM`, or `MRP` followed by a digit → small).

**Overlays.** `mtDNA_encoded` and `mito` are recorded as separate boolean
columns rather than classes, so that mtDNA-encoded products and the
Morgenstern mitochondrial set can be marked independently of the functional
colouring.

In [ ]:
# --- build MITO_GENES from the Morgenstern high-confidence proteome ---
import ast
ANNO_FILE = f"{DATA_DIR}/yeast_encyclopedia.pkl"   # <-- annotation table


anno = pd.read_pickle(ANNO_FILE)    # MultiIndex columns come back intact

def _norm_col(c):
    """(level0, level1) whether c is a tuple, a "('a','b')" string, or a plain str."""
    if isinstance(c, tuple):
        return (str(c[0]), str(c[1]) if len(c) > 1 else "")
    s = str(c)
    if s.startswith("(") and "," in s:
        try:
            t = ast.literal_eval(s)
            if isinstance(t, tuple):
                return (str(t[0]), str(t[1]) if len(t) > 1 else "")
        except Exception:
            pass
    return (s, "")

def _find_col(df, l0, l1):
    for c in df.columns:
        a, b = _norm_col(c)
        if l0.lower() in a.lower() and l1.lower() in b.lower():
            return c
    raise KeyError(f"no column matching ({l0!r}, {l1!r})")

gene_col = _find_col(anno, "GeneName", "")
morg_col = _find_col(anno, "Morgenstern", "High confidence")

# --- diagnostics: run once to confirm the flag column looks right ---
print("gene_col =", gene_col, "| morg_col =", morg_col)
print(anno[morg_col].value_counts(dropna=False))

# --- robust truthiness: handles bool True, "True", 1, NaN, blank ---
col = anno[morg_col]
if col.dtype == bool:
    flag = col.fillna(False)
else:
    flag = col.astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes", "y", "+"])

MITO_GENES = set(
    anno.loc[flag, [gene_col]][gene_col]   # [gene_col] wraps tuple -> single column selector
        .dropna().astype(str)
        .str.upper().str.split(";").str[0].str.strip()
) - {"", "NAN"}
print(f"MITO_GENES: {len(MITO_GENES)} genes")


In [ ]:
# --- protein classes (edit these gene lists to fit the manuscript) ---
CLASSES = {


    
    "PDH complex":            ["PDA1","PDB1","LAT1","LPD1","PDX1"],
    "mtDNA / nucleoid":       [ "ABF2","ACO1", "ATP1","CHA1","ECM10","HSP60","IDH1","IDP1","ILV5","ILV6",
                                "KGD1","KGD2","LPD1","LSC1","MGM101","MNP1","PDA1","PDB1","RIM1","RPO41","SLS1",
                                "SSC1","YHM2", "MIP1", "MSH1", "CCE1", "IRC3"],
    "Large Ribosomal Subunit": ["IMG1","IMG2","MHR1","MNP1","MRP20","MRP35","MRP49","MRP7",
                                    "MRPL1","MRPL10","MRPL11","MRPL13","MRPL15","MRPL16","MRPL17","MRPL19",
                                    "MRPL20","MRPL22","MRPL23","MRPL24","MRPL25","MRPL27","MRPL28","MRPL3",
                                    "MRPL31","MRPL32","MRPL33","MRPL35","MRPL36","MRPL37","MRPL38","MRPL39",
                                    "MRPL4","MRPL40","MRPL44","MRPL49","MRPL50","MRPL51","MRPL6","MRPL7",
                                    "MRPL8","MRPL9","MRX14","PTH4","RML2","RTC6","YML6",],  # matched by prefix below (MRPL/MRPS/RSM/MRP)
    "Small Ribosomal Subunit": ["EHD3","FYV4","MRP1","MRP10","MRP13","MRP17","MRP2","MRP21","MRP4","MRP51",
                                "MRPS12","MRPS16","MRPS17","MRPS18","MRPS28","MRPS35","MRPS5","MRPS8","MRPS9",
                                "NAM9","PET123","PPE1","QRI5","RSM10","RSM18","RSM19","RSM22","RSM23","RSM24",
                                "RSM25","RSM26","RSM27","RSM28","RSM7","SWS2","VAR1"],
    "TCA":["ACO1","ACO2","CIT1","CIT2","CIT3","FUM1","IDH1","IDH2","IDP1","IDP2","IDP3",
            "KGD1","KGD2","KGD4","LPD1","LSC1","LSC2","MDH1","MDH2","MDH3","SDH1","SDH2",
            "SDH3","SDH4","SDH5","SDH9","SHH3","SHH4"],
    
    "Trans. Factors":['RSO55', 'SLS1', 'HER2', 'DPC29', 'SOV1', 'MEF2', 'NAM2', 'MEF1', 'RRF1', 'IFM1', 'PET54', 'CCM1',
                      'MSR1', 'ISM1', 'MMF1', 'GTF1', 'MTF2', 'HTS1', 'PET112', 'MSK1', 'MRF1', 'MSE1', 'TUF1', 'PTH1', 'PET127','AEP3','NAM1'],
    "mtDNA encoded":['COX1', 'COX2', 'COX3', 'ATP6', 'ATP8', 'ATP9', 'COB', 'VAR1'],
}

# ----- plumbing derived from CLASSES so every class is drawn/coloured/tested -----
CLASS_ORDER = list(CLASSES.keys()) + ["other"]
_PALETTE = ["#d1495b","#2a9d8f","#e9773b","#edb230","#8c6bb1","#5a7ec9","#b07aa1","#76b041"]
COLORS = {cls: _PALETTE[i % len(_PALETTE)] for i, cls in enumerate(CLASSES.keys())}
COLORS["other"] = "#c9c9c9"

def classify(g):
    for cls, genes in CLASSES.items():
        if g in genes:
            return cls
    # fallback for any ribosomal proteins not explicitly listed
    if g.startswith("MRPL"):
        return "Large Ribosomal Subunit"
    if g.startswith(("MRPS","RSM")) or (g.startswith("MRP") and g[3:4].isdigit()):
        return "Small Ribosomal Subunit"
    return "other"

res["class"] = res["gene"].map(classify)

# mtDNA-encoded products (nascent chains) — annotate separately (overlay, not a class)
MTDNA_ENCODED = {"COX1","COX2","COX3","COB","CYTB","ATP6","ATP8","ATP9","OLI1","VAR1"}
res["mtDNA_encoded"] = res["gene"].isin(MTDNA_ENCODED)

# optional mitochondrial reference (Morgenstern 2017 / SGD GO:0005739) for the mito-fraction test
#MITO_GENES = set()   # <-- load your yeast mito reference here to enable that test
res["mito"] = res["gene"].isin(MITO_GENES) if MITO_GENES else np.nan

print(res["class"].value_counts())
print("mtDNA-encoded detected:", sorted(res.loc[res["mtDNA_encoded"], "gene"].tolist()))

## 6. Narrative volcano and top-interactor table

Collapses the fine-grained functional classes into the broader groups used in
the manuscript figure, then generates the volcano plot and a ranked
interactor table.

### Display groups

The classes from Section 5 are mapped onto five narrative groups: the three
mitoribosome and translation-factor classes are merged into a single
**Mitochondrial translation** group, while PDH complex, nucleoid/mtDNA
maintenance, and mtDNA-encoded products are retained separately. Proteins
without a functional class are split by the Morgenstern mitochondrial flag into
**other mitochondrial** and **non-mitochondrial**.

### Volcano plot

- **Draw order and styling** run background-to-foreground: non-mitochondrial
  proteins are small, pale and semi-transparent; story groups are progressively
  larger, more opaque, and drawn on top. This keeps the narrative visible
  without hiding the full distribution.
- **mtDNA-encoded products** are ringed in black wherever they fall, marking
  them independently of their group colour.
- **Threshold guides** are drawn at q = 0.05 and ±log2FC = 1.
- **Labelling** is restricted to significant proteins in the PDH complex,
  nucleoid, and mtDNA-encoded groups. Mitochondrial translation hits are capped
  at the top 8 by π-score to keep the panel legible; proteins in the background
  groups are never labelled. `adjustText`, if installed, repositions labels to
  avoid overlap.
- **x-axis limits** are set from the 99th percentile of |log2FC| with 15%
  padding, so extreme outliers do not compress the informative region.

### π-score ranking

Proteins are ranked by π-score (log2 fold change × −log10 q), which combines
effect size and significance rather than thresholding on either alone. The
printed summary breaks the top 30 down by functional group and reports the
mitochondrial fraction.

### Interactor table figure

Renders the same top-30 ranking as a standalone figure panel, grouped by
display group and colour-matched to the volcano, with log2 fold change and
q-value per protein. Header row text colour is chosen automatically by
background luminance.

In [ ]:
# --- collapse fine classes into the manuscript narrative groups ---
STORY_MAP = {
    "PDH complex":              "PDH complex",
    "mtDNA / nucleoid":         "Nucleoid / mtDNA maintenance",
    "Large Ribosomal Subunit":  "Mitochondrial translation",
    "Small Ribosomal Subunit":  "Mitochondrial translation",
    "Trans. Factors":           "Mitochondrial translation",
    "mtDNA encoded":            "mtDNA-encoded (nascent)",
}
_story   = res["class"].map(STORY_MAP)
_is_mito = res["mito"].fillna(False).astype(bool) if res["mito"].notna().any() else pd.Series(False, index=res.index)
res["display_group"] = np.where(_story.notna(), _story,
                          np.where(_is_mito, "other mitochondrial", "non-mitochondrial"))

# draw order (background first) and colours
DISPLAY_ORDER  = ["non-mitochondrial", "other mitochondrial", "Mitochondrial translation",
                  "Nucleoid / mtDNA maintenance", "mtDNA-encoded (nascent)", "PDH complex"]
DISPLAY_COLORS = {"non-mitochondrial":"#dddddd", "other mitochondrial":"#9aabb8",
                  "Mitochondrial translation":"#e9773b", "Nucleoid / mtDNA maintenance":"#2a9d8f",
                  "mtDNA-encoded (nascent)":"#7d4fa1", "PDH complex":"#d1495b"}
print(res["display_group"].value_counts().to_string())

In [ ]:
# --- narrative volcano: dim contaminants, highlight the story ---
import warnings
from matplotlib.lines import Line2D
try:
    from adjustText import adjust_text
except ImportError:
    adjust_text = None
    warnings.warn("adjustText not installed -> labels may overlap. Run: pip install adjustText")

if "pi" not in res.columns:
    res["pi"] = res["log2FC"] * res["neglog10q"]

fig, ax = plt.subplots(figsize=(7.6, 6.2))

# per-group point styling (size, alpha, z-order, edge)
STYLE = {
    "non-mitochondrial":            dict(s=10, a=0.45, z=1, ec="none"),
    "other mitochondrial":          dict(s=22, a=0.75, z=2, ec="white"),
    "Mitochondrial translation":    dict(s=42, a=0.95, z=4, ec="white"),
    "Nucleoid / mtDNA maintenance": dict(s=46, a=0.95, z=5, ec="white"),
    "mtDNA-encoded (nascent)":      dict(s=46, a=0.95, z=5, ec="white"),
    "PDH complex":                  dict(s=52, a=0.98, z=6, ec="white"),
}
for grp in DISPLAY_ORDER:
    sub = res[res["display_group"] == grp]
    if sub.empty: continue
    st = STYLE[grp]
    ax.scatter(sub["log2FC"], sub["neglog10q"], s=st["s"], c=DISPLAY_COLORS[grp],
               alpha=st["a"], edgecolor=st["ec"], linewidth=0.4, zorder=st["z"], label=grp)

# ring the mtDNA-encoded nascent chains (Cox1/2/3, Var1 ...) wherever they sit
mt = res[res["mtDNA_encoded"]]
if not mt.empty:
    ax.scatter(mt["log2FC"], mt["neglog10q"], s=95, facecolor="none",
               edgecolor="black", linewidth=1.0, zorder=7)

# threshold guides
ax.axhline(-np.log10(Q_CUT), ls="--", lw=0.8, c="0.6")
ax.axvline( FC_CUT, ls="--", lw=0.8, c="0.6")
ax.axvline(-FC_CUT, ls="--", lw=0.8, c="0.6")

# labels: story proteins only (never contaminants / background); cap translation
STORY_GROUPS = ["PDH complex", "Nucleoid / mtDNA maintenance", "mtDNA-encoded (nascent)"]
lab    = res[res["significant"] & res["display_group"].isin(STORY_GROUPS)].copy()
transl = res[res["significant"] & (res["display_group"] == "Mitochondrial translation")].nlargest(8, "pi")
lab = pd.concat([lab, transl]).drop_duplicates("gene")
lab = lab[lab["gene"].notna() & (lab["gene"] != "NAN")]
texts = [ax.text(r["log2FC"], r["neglog10q"], r["gene"], fontsize=7.5, fontstyle="italic", zorder=8)
         for _, r in lab.iterrows()]
if adjust_text:
    adjust_text(texts, ax=ax, x=res["log2FC"].values, y=res["neglog10q"].values,
                force_text=(0.5, 0.9), expand=(1.3, 1.8), max_move=(30, 30),
                arrowprops=dict(arrowstyle="-", color="0.55", lw=0.5))

ax.set_xlabel("log$_2$ fold change  (Lat1 / WT)")
ax.set_ylabel("$-$log$_{10}$ $q$-value")
ax.set_title("Lat1 pull-down interactome")
xm = np.nanpercentile(np.abs(res["log2FC"]), 99) * 1.15
ax.set_xlim(-xm, xm)

handles, _ = ax.get_legend_handles_labels()
handles.append(Line2D([0],[0], marker="o", linestyle="none", markerfacecolor="none",
                       markeredgecolor="black", markersize=8, label="mtDNA-encoded"))
ax.legend(handles=handles, frameon=False, fontsize=7, loc="upper left")
fig.tight_layout()
#fig.savefig("volcano_lat1_narrative.pdf", bbox_inches="tight")
#fig.savefig("volcano_lat1_narrative.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved volcano_lat1_narrative.pdf / .png")

In [ ]:
# --- statement of the most enriched interactors (descriptive; no stats claim) ---
if "pi" not in res.columns:
    res["pi"] = res["log2FC"] * res["neglog10q"]

TOPN = 30
top = res[res["significant"]].sort_values("pi", ascending=False).head(TOPN).copy()
N = len(top)   # actual count (may be < TOPN if fewer significant proteins)
_mito = top["mito"].fillna(False).astype(bool)
_m = lambda mask: top.loc[mask, "gene"].tolist()

n_mito   = int(_mito.sum())
contam   = _m(~_mito)
pdh      = _m(top["class"] == "PDH complex")
nucl     = _m(top["class"] == "mtDNA / nucleoid")
transl   = _m(top["display_group"] == "Mitochondrial translation")
mtenc    = _m(top["mtDNA_encoded"])
other    = _m(top["display_group"] == "other mitochondrial")

j = lambda xs: ", ".join(xs) if xs else "—"
print(f"Top {N} interactors by π-score (log2FC × −log10 q)")
print("=" * 60)
print(f"mitochondrial: {n_mito}/{N}   non-mitochondrial: {N-n_mito}")
print(f"  PDH complex ({len(pdh)}): {j(pdh)}")
print(f"  Nucleoid / mtDNA maintenance ({len(nucl)}): {j(nucl)}")
print(f"  Mitochondrial translation ({len(transl)}): {j(transl)}")
print(f"  mtDNA-encoded / nascent ({len(mtenc)}): {j(mtenc)}")
print(f"  other abundant mitochondrial ({len(other)}): {j(other)}")
print(f"  non-mitochondrial (contaminants): {j(contam)}")



In [ ]:
# --- top-N interactor table figure (reuses the volcano DISPLAY_COLORS) ---
from matplotlib.patches import Rectangle

def _text_on(hexc):
    h = hexc.lstrip("#"); r, g, b = [int(h[k:k+2], 16)/255 for k in (0, 2, 4)]
    return "white" if 0.299*r + 0.587*g + 0.114*b < 0.6 else "#222222"

def make_top_table(res, TOPN=30, fname="lat1_top_interactors_table",
                   group_order=("PDH complex", "Nucleoid / mtDNA maintenance",
                                "Mitochondrial translation", "mtDNA-encoded (nascent)",
                                "other mitochondrial", "non-mitochondrial")):
    if "pi" not in res.columns:
        res["pi"] = res["log2FC"] * res["neglog10q"]
    top = res[res["significant"]].sort_values("pi", ascending=False).head(TOPN)

    seq = []
    for g in group_order:
        sub = top[top["display_group"] == g].sort_values("pi", ascending=False)
        if sub.empty: continue
        seq.append(("H", g, len(sub)))
        for _, r in sub.iterrows():
            seq.append(("R", r["gene"], r["log2FC"], r["q"], g))
    n = len(seq)

    fig, ax = plt.subplots(figsize=(3.9, 0.34*n + 0.9))
    ax.set_xlim(0, 1); ax.set_ylim(0, n + 1.5); ax.axis("off")
    xg, xfc, xq = 0.10, 0.66, 0.99
    ax.text(0.5, n+1.05, f"Top {TOPN} Lat1 interactors ($\\pi$-ranked)",
            fontsize=9.5, fontweight="bold", ha="center")
    ax.text(xg,  n+0.3, "protein",  fontsize=7, color="#888", ha="left", style="italic")
    ax.text(xfc, n+0.3, "log$_2$FC", fontsize=7, color="#888", ha="right")
    ax.text(xq,  n+0.3, "q",         fontsize=7, color="#888", ha="right")

    y = n
    for item in seq:
        y -= 1
        if item[0] == "H":
            _, g, k = item; col = DISPLAY_COLORS[g]
            ax.add_patch(Rectangle((0, y), 1, 1, facecolor=col, edgecolor="none"))
            ax.text(0.03, y+0.5, f"{g}  ({k})", fontsize=7.5, fontweight="bold",
                    color=_text_on(col), va="center")
        else:
            _, gene, fc, q, g = item; col = DISPLAY_COLORS[g]
            ax.add_patch(Rectangle((0, y), 1, 1, facecolor=col, alpha=0.14, edgecolor="none"))
            ax.add_patch(Rectangle((0, y), 0.022, 1, facecolor=col, edgecolor="none"))  # left swatch
            ax.text(xg,  y+0.5, str(gene), fontsize=8, style="italic", va="center")
            ax.text(xfc, y+0.5, f"{fc:.1f}", fontsize=7.5, color="#333", va="center", ha="right")
            ax.text(xq,  y+0.5, f"{q:.0e}",  fontsize=7,   color="#777", va="center", ha="right")

    fig.tight_layout()
    fig.savefig(fname + ".pdf", bbox_inches="tight")
    fig.savefig(fname + ".png", dpi=300, bbox_inches="tight")
    plt.show(); print("saved", fname + ".pdf / .png")
    return fig

make_top_table(res, TOPN=30)

## 7. Export results table

Writes the complete per-protein results to CSV, sorted with significant
enrichments first and then by decreasing significance within each block. The
exported table carries the protein group IDs, gene name, log2 fold change,
raw p-value, BH q-value, functional class, display group, and the
mitochondrial and mtDNA-encoded flags for every protein passing the
valid-value filter.

The first 25 rows are displayed inline as a check on the ranking.

In [ ]:
# --- export the results table (sorted by significance) ---
out = res.sort_values(["significant","neglog10q"], ascending=[False,False])
out.to_csv(f"{OUT_DIR}/lat1_volcano_results.csv", index=False)
out.head(25)